In [1]:
import torch
import torch.nn.functional as F

d:\ComputerScience\BachKhoa\ProjectII\YOLOQA\venv\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
from model import QASpanDetector
# model = torch.load('models/yolo_qa_v3.pth', map_location=torch.device('cpu'))
model = QASpanDetector()

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForTokenClassification: ['cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-u

In [3]:
from transformers.data.metrics.squad_metrics import compute_f1

In [5]:
import pickle

with open('data/train_preprocessed2.pkl', 'rb') as f:
    train_encodings = pickle.load(f)

In [6]:
from preprocess import SquadDataset

train_dataset = SquadDataset(train_encodings)

In [7]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False)

In [8]:
MODEL_MAX_LENGTH = 512

In [9]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [26]:
import csv

device = "cuda" if torch.cuda.is_available() else 'cpu'

model.eval()
f1 = 0
num_examples = 0

# open the file in the write mode
f = open('results.csv', 'w')
writer = csv.writer(f)
writer.writerow(['Ground Truth', 'Prediction', 'F1'])

print("############Evaluate############")
with torch.no_grad():
    for batch_idx, batch in enumerate(train_loader):
        sentence_length = batch['input_ids'].size(1)
        batch_size = batch['input_ids'].size(0)

        answer_start = int(batch['start_positions'])
        if answer_start >= MODEL_MAX_LENGTH:
            continue

        answer_end = int(batch['end_positions'])
        answer_target = tokenizer.decode(batch['input_ids'][0, answer_start:answer_end+1])

        outputs = model(batch['input_ids'], batch['attention_mask'])
        obj_pred = outputs['logits'][0, :, 0]
        answer_start_pred = int(torch.argmax(obj_pred))
        answer_length_pred = int(outputs['logits'][0, :, 1][answer_start_pred].exp().round())
        answer_pred = tokenizer.decode(batch['input_ids'][0, answer_start_pred:answer_start_pred+answer_length_pred])
        
        f1_score = compute_f1(answer_target, answer_pred)
        writer.writerow([answer_target, answer_pred, f1_score])

        f1 += f1_score
        num_examples += 1
        
        break

print('Average F1:', f1 / num_examples)
f.close()


############Evaluate############
Average F1: 0.0
